In [1]:
import pandas as pd
import numpy as np

In [57]:
RESULT_PATH = "D:\\IE403\\ViAIGS\\outputs\\llm_judge_score_rule\\final_score_rule_results.csv"
OUTPUT_DIR = "D:\\IE403\\ViAIGS\\outputs\\llm_judge_analysis"

In [58]:
df = pd.read_csv(RESULT_PATH)
df.head()

,judge_item_id,original_id,candidate_id,generator,source,original_text,candidate_text,surface_similarity,empty_original,empty_candidate,...,decision,final_decision,reason,raw_judge_response,judge_error,missing_score_count,quality_total_score,quality_avg_score,score_rule_decision,score_rule_reason
0,aya::human_013660_para_r1,human_013660,human_013660_para_r1,aya,fb,Nguyễn Bích Như học tập đi,"Nâng niu mơ ước, Nguyễn Bích Như hãy cố gắng h...",0.5455,False,False,...,drop,drop,Dropped by rule-based precheck.,NaN,NaN,5,0.0,0.0,drop,total_score_0_le_12_drop
1,aya::human_025981_para_r3,human_025981,human_025981_para_r3,aya,ytb,Trên đất nước nghèo đói của Việt Nam có mấy n...,"Trên đất nước Việt Nam nghèo khó, có những cá ...",0.2016,False,False,...,drop,drop,Dropped by rule-based precheck.,NaN,NaN,5,0.0,0.0,drop,total_score_0_le_12_drop
2,aya::human_028119_para_r2,human_028119,human_028119_para_r2,aya,ytb,Mình giới thiệu 1 cuốn sách trinh thám kinh đi...,"Cuốn sách trinh thám kinh điển, ra mắt vào năm...",0.3071,False,False,...,drop,drop,Dropped by rule-based precheck.,NaN,NaN,5,0.0,0.0,drop,total_score_0_le_12_drop
3,aya::human_016713_para_r3,human_016713,human_016713_para_r3,aya,ytb,Mua dép thì phải làm như thế nào ạ?,"Khi mua dép, bạn cần lưu ý những điều sau đây ...",0.3765,False,False,...,drop,drop,Dropped by rule-based precheck.,NaN,NaN,5,0.0,0.0,drop,total_score_0_le_12_drop
4,aya::human_014594_para_r3,human_014594,human_014594_para_r3,aya,fb,Thảo Uyên Nguyễn,Thảo Uyên Nguyễn vừa bày tỏ tâm trạng mới nhất...,0.3556,False,False,...,drop,drop,Dropped by rule-based precheck.,NaN,NaN,5,0.0,0.0,drop,total_score_0_le_12_drop


In [59]:
df.columns

Index(['judge_item_id', 'original_id', 'candidate_id', 'generator', 'source',
       'original_text', 'candidate_text', 'surface_similarity',
       'empty_original', 'empty_candidate', 'ai_artifact_flag',
       'encoding_noise_flag', 'too_short_flag', 'too_similar_flag',
       'rule_predecision', 'length', 'candidate_length', 'paraphrase_round',
       'generation_method', 'bleu_score', 'bertscore', 'crossencoder_score',
       'ld', 'edit_similarity', 'meaning_score', 'naturalness_score',
       'social_style_score', 'language_consistency_score', 'ai_artifact_score',
       'dataset_usefulness_score', 'overall_quality_score', 'decision',
       'final_decision', 'reason', 'raw_judge_response', 'judge_error',
       'missing_score_count', 'quality_total_score', 'quality_avg_score',
       'score_rule_decision', 'score_rule_reason'],
      dtype='object')

In [60]:
score_cols = [
    "meaning_score",
    "naturalness_score",
    "social_style_score",
    "language_consistency_score",
    "dataset_usefulness_score",
]

In [61]:
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [62]:
def safe_rate(series, value):
    return (series == value).mean()

In [63]:
def build_overall_summary(df):
    total = len(df)

    decision_col = "score_rule_decision"

    keep_rate = safe_rate(df[decision_col], "keep")
    review_rate = safe_rate(df[decision_col], "review")
    drop_rate = safe_rate(df[decision_col], "drop")

    avg_total_score = df["quality_total_score"].mean()
    avg_quality_score = df["quality_avg_score"].mean()

    avg_meaning = df["meaning_score"].mean()
    avg_natural = df["naturalness_score"].mean()
    avg_social = df["social_style_score"].mean()
    avg_language = df["language_consistency_score"].mean()
    avg_usefulness = df["dataset_usefulness_score"].mean()

    quality_index = (
        0.40 * keep_rate
        + 0.30 * (avg_total_score / 25)
        + 0.15 * (1 - drop_rate)
        + 0.15 * (avg_quality_score / 5)
    ) * 100

    summary = {
        "total_samples": total,
        "keep_samples": (df[decision_col] == "keep").sum(),
        "review_samples": (df[decision_col] == "review").sum(),
        "drop_samples": (df[decision_col] == "drop").sum(),
        "keep_rate": keep_rate,
        "review_rate": review_rate,
        "drop_rate": drop_rate,
        "avg_total_score_25": avg_total_score,
        "avg_quality_score_5": avg_quality_score,
        "avg_meaning_score": avg_meaning,
        "avg_naturalness_score": avg_natural,
        "avg_social_style_score": avg_social,
        "avg_language_consistency_score": avg_language,
        "avg_dataset_usefulness_score": avg_usefulness,
        "avg_ai_artifact_score_reference": df["ai_artifact_score"].mean() if "ai_artifact_score" in df else np.nan,
        "avg_overall_quality_score_reference": df["overall_quality_score"].mean() if "overall_quality_score" in df else np.nan,
        "ai_artifact_flag_rate": df["ai_artifact_flag"].mean() if "ai_artifact_flag" in df else np.nan,
        "encoding_noise_rate": df["encoding_noise_flag"].mean() if "encoding_noise_flag" in df else np.nan,
        "too_short_rate": df["too_short_flag"].mean() if "too_short_flag" in df else np.nan,
        "too_similar_rate": df["too_similar_flag"].mean() if "too_similar_flag" in df else np.nan,
        "judge_error_rate": df["judge_error"].notna().mean() if "judge_error" in df else np.nan,
        "dataset_quality_index": quality_index,
    }

    return pd.DataFrame([summary])

In [64]:
overall_summary = build_overall_summary(df)

overall_summary.to_csv(
    f"{OUTPUT_DIR}/overall_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

overall_summary

,total_samples,keep_samples,review_samples,drop_samples,keep_rate,review_rate,drop_rate,avg_total_score_25,avg_quality_score_5,avg_meaning_score,...,avg_language_consistency_score,avg_dataset_usefulness_score,avg_ai_artifact_score_reference,avg_overall_quality_score_reference,ai_artifact_flag_rate,encoding_noise_rate,too_short_rate,too_similar_rate,judge_error_rate,dataset_quality_index
0,9001,3867,765,4369,0.429619,0.084991,0.485391,11.822797,2.364559,2.192645,...,2.585157,2.336851,1.296156,4.223272,0.0,0.465615,0.0,0.002444,0.005444,46.184935


In [69]:
def add_quality_level(score):
    if score >= 75:
        return "High"
    elif score >= 55:
        return "Good"
    elif score >= 45:
        return "Acceptable"
    else:
        return "Low"


decision_col = "score_rule_decision"

generator_summary = (
    df.groupby("generator")
    .agg(
        total=("judge_item_id", "count"),
        keep=(decision_col, lambda x: (x == "keep").sum()),
        review=(decision_col, lambda x: (x == "review").sum()),
        drop=(decision_col, lambda x: (x == "drop").sum()),
        keep_rate=(decision_col, lambda x: (x == "keep").mean()),
        review_rate=(decision_col, lambda x: (x == "review").mean()),
        drop_rate=(decision_col, lambda x: (x == "drop").mean()),
        avg_total_score=("quality_total_score", "mean"),
        avg_quality_score=("quality_avg_score", "mean"),
        avg_meaning=("meaning_score", "mean"),
        avg_naturalness=("naturalness_score", "mean"),
        avg_social_style=("social_style_score", "mean"),
        avg_language=("language_consistency_score", "mean"),
        avg_usefulness=("dataset_usefulness_score", "mean"),
        avg_ai_artifact_reference=("ai_artifact_score", "mean"),
        avg_overall_quality_reference=("overall_quality_score", "mean"),
        judge_error_rate=("judge_error", lambda x: x.notna().mean() if x.notna().any() else 0),
    )
    .reset_index()
)

generator_summary["quality_index"] = (
    0.40 * generator_summary["keep_rate"]
    + 0.30 * (generator_summary["avg_total_score"] / 25)
    + 0.15 * (generator_summary["avg_quality_score"] / 5)
    + 0.15 * (1 - generator_summary["drop_rate"])
) * 100

generator_summary["quality_level"] = generator_summary["quality_index"].apply(add_quality_level)

generator_summary = generator_summary.sort_values("quality_index", ascending=False)

generator_summary.to_csv(
    f"{OUTPUT_DIR}/generator_quality_summary_score_rule.csv",
    index=False,
    encoding="utf-8-sig"
)

generator_summary

,generator,total,keep,review,drop,keep_rate,review_rate,drop_rate,avg_total_score,avg_quality_score,avg_meaning,avg_naturalness,avg_social_style,avg_language,avg_usefulness,avg_ai_artifact_reference,avg_overall_quality_reference,judge_error_rate,quality_index,quality_level
3,gpt,2064,1108,57,899,0.536822,0.027616,0.435562,13.406977,2.681395,2.635174,2.457364,2.803295,2.823159,2.687984,1.337329,4.632705,0.007752,54.071996,Acceptable
0,aya,1700,759,128,813,0.446471,0.075294,0.478235,12.057647,2.411529,2.251765,2.221765,2.582353,2.616471,2.385294,1.331504,4.262349,0.006471,47.389059,Acceptable
2,gemma,1614,668,207,739,0.413879,0.128253,0.457869,12.141884,2.428377,2.156753,2.233581,2.647460,2.726146,2.377943,1.237832,3.991150,0.001239,46.542503,Acceptable
1,deepseek,1897,769,133,995,0.405377,0.070111,0.524512,10.942014,2.188403,2.050079,1.980496,2.357934,2.385872,2.167633,1.172638,4.270358,0.007380,43.043015,Low
4,sailor,1726,563,240,923,0.326188,0.139050,0.534762,10.366744,2.073349,1.795481,1.909038,2.288528,2.356895,2.016802,1.396733,3.817970,0.003476,38.686211,Low


In [70]:
decision_col = "score_rule_decision"

source_summary = (
    df.groupby("source")
    .agg(
        total=("judge_item_id", "count"),
        keep_rate=(decision_col, lambda x: (x == "keep").mean()),
        review_rate=(decision_col, lambda x: (x == "review").mean()),
        drop_rate=(decision_col, lambda x: (x == "drop").mean()),
        avg_total_score=("quality_total_score", "mean"),
        avg_quality_score=("quality_avg_score", "mean"),
        avg_meaning=("meaning_score", "mean"),
        avg_naturalness=("naturalness_score", "mean"),
        avg_social_style=("social_style_score", "mean"),
        avg_language=("language_consistency_score", "mean"),
        avg_usefulness=("dataset_usefulness_score", "mean"),
        too_short_rate=("too_short_flag", "mean"),
        too_similar_rate=("too_similar_flag", "mean"),
    )
    .reset_index()
)

source_summary["quality_index"] = (
    0.40 * source_summary["keep_rate"]
    + 0.30 * (source_summary["avg_total_score"] / 25)
    + 0.15 * (source_summary["avg_quality_score"] / 5)
    + 0.15 * (1 - source_summary["drop_rate"])
) * 100

source_summary["quality_level"] = source_summary["quality_index"].apply(add_quality_level)

source_summary = source_summary.sort_values("quality_index", ascending=False)

source_summary.to_csv(
    f"{OUTPUT_DIR}/source_quality_summary_score_rule.csv",
    index=False,
    encoding="utf-8-sig"
)

source_summary

,source,total,keep_rate,review_rate,drop_rate,avg_total_score,avg_quality_score,avg_meaning,avg_naturalness,avg_social_style,avg_language,avg_usefulness,too_short_rate,too_similar_rate,quality_index,quality_level
2,ytb,3069,0.476377,0.090909,0.432714,13.177908,2.635582,2.475073,2.438579,2.801564,2.850114,2.612577,0.0,0.002281,51.284588,Acceptable
0,fb,2826,0.464614,0.089526,0.445860,12.718684,2.543737,2.329441,2.345011,2.743454,2.786624,2.514154,0.0,0.001062,49.790304,Acceptable
1,voz,3106,0.351578,0.075016,0.573406,9.668706,1.933741,1.789118,1.736961,2.099485,2.140052,1.903091,0.0,0.003863,37.865679,Low


In [71]:
decision_col = "score_rule_decision"

generator_source_summary = (
    df.groupby(["generator", "source"])
    .agg(
        total=("judge_item_id", "count"),
        keep_rate=(decision_col, lambda x: (x == "keep").mean()),
        review_rate=(decision_col, lambda x: (x == "review").mean()),
        drop_rate=(decision_col, lambda x: (x == "drop").mean()),
        avg_total_score=("quality_total_score", "mean"),
        avg_quality_score=("quality_avg_score", "mean"),
        avg_social_style=("social_style_score", "mean"),
        avg_meaning=("meaning_score", "mean"),
    )
    .reset_index()
)

generator_source_summary["quality_index"] = (
    0.40 * generator_source_summary["keep_rate"]
    + 0.30 * (generator_source_summary["avg_total_score"] / 25)
    + 0.15 * (generator_source_summary["avg_quality_score"] / 5)
    + 0.15 * (1 - generator_source_summary["drop_rate"])
) * 100

generator_source_summary["quality_level"] = generator_source_summary["quality_index"].apply(add_quality_level)

generator_source_summary.to_csv(
    f"{OUTPUT_DIR}/generator_source_quality_summary_score_rule.csv",
    index=False,
    encoding="utf-8-sig"
)

generator_source_summary.sort_values("quality_index", ascending=False)

,generator,source,total,keep_rate,review_rate,drop_rate,avg_total_score,avg_quality_score,avg_social_style,avg_meaning,quality_index,quality_level
11,gpt,ytb,685,0.594161,0.023358,0.382482,14.827737,2.965547,3.065693,2.941606,59.719124,Good
9,gpt,fb,659,0.552352,0.036419,0.411229,13.916540,2.783308,2.928680,2.702580,55.975417,Good
6,gemma,fb,493,0.472617,0.152130,0.375254,13.987830,2.797566,3.083164,2.434077,53.453955,Acceptable
2,aya,ytb,570,0.489474,0.084211,0.426316,13.457895,2.691579,2.856140,2.549123,52.408421,Acceptable
8,gemma,ytb,563,0.463588,0.122558,0.413854,13.303730,2.660746,2.852575,2.399645,51.282416,Acceptable
0,aya,fb,518,0.480695,0.081081,0.438224,12.866795,2.573359,2.750965,2.366795,50.814672,Acceptable
5,deepseek,ytb,646,0.475232,0.083591,0.441176,12.947368,2.589474,2.755418,2.455108,50.696904,Acceptable
10,gpt,voz,720,0.468056,0.023611,0.508333,11.588889,2.317778,2.438889,2.281944,46.957222,Acceptable
3,deepseek,fb,591,0.423012,0.065990,0.510998,11.294416,2.258883,2.448393,2.091371,44.585448,Low
12,sailor,fb,565,0.384071,0.129204,0.486726,11.568142,2.313628,2.532743,2.017699,43.884602,Low


In [72]:
flag_cols = [
    "empty_original",
    "empty_candidate",
    "ai_artifact_flag",
    "encoding_noise_flag",
    "too_short_flag",
    "too_similar_flag",
]

issue_summary = []

for col in flag_cols:
    if col in df.columns:
        temp = df[col]

        if temp.dtype == "object":
            temp = temp.astype(str).str.lower().isin(["true", "1", "yes"])

        issue_summary.append({
            "issue_type": col,
            "count": temp.sum(),
            "rate": temp.mean(),
        })

issue_summary = pd.DataFrame(issue_summary).sort_values("count", ascending=False)

issue_summary.to_csv(
    f"{OUTPUT_DIR}/issue_summary_score_rule.csv",
    index=False,
    encoding="utf-8-sig"
)

issue_summary

,issue_type,count,rate
3,encoding_noise_flag,4191,0.465615
5,too_similar_flag,22,0.002444
1,empty_candidate,0,0.000000
0,empty_original,0,0.000000
2,ai_artifact_flag,0,0.000000
4,too_short_flag,0,0.000000
